# 5.4 · Self-balancing trees: red-black trees and treaps

*Trees* · 15 exercises

Keep a search tree's height logarithmic whatever the insert order: colour rules, or random priorities.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-5.4) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-5.4) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-5.4)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import math
import random

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

In [ ]:
# Support code used by the exercises: run it as is.
RED, BLACK = "R", "B"

### 5.4.1 · Nodes, the NIL sentinel and rotations  `Core`

**Task.** Create `RBNode(key, color)` with `left`, `right` and `parent`, a `RedBlackTree` whose empty children all point at one black `NIL` sentinel, and `rotate_left(x)` / `rotate_right(x)` that keep parent links and the root correct.

**Example.** after a left rotation at x, x's old right child takes x's place and x becomes its left child; in-order order is unchanged

**Big O.** O(1) per rotation: three links change.

**Note.** A rotation never breaks binary-search order. It only changes the shape, which is all balancing needs.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-1)

### 5.4.2 · Insert with recolouring and rotations  `Core`

**Task.** Add `insert(key)`: do a normal BST insert of a red node, then fix up. While the parent is red: if the uncle is red, recolour parent, uncle and grandparent and move up; otherwise rotate (once or twice) and recolour. Finally paint the root black.

**Example.** insert 1..7 in sorted order -> root `2`, height `4` (a plain BST would be 7 tall), and `validate()` passes after every insert

**Big O.** O(log N): the walk down is O(log N), fix-up recolours up the tree and does at most two rotations.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-2)

### 5.4.3 · Check the red-black rules  `Core`

**Task.** Add `validate()` returning the tree's black height, and raising `AssertionError` if the root is red, a red node has a red child, BST order is broken, or two paths have different black counts.

**Example.** after any sequence of inserts and deletes, `validate()` returns a positive number instead of raising

**Big O.** O(N). Write this before insert: it is the test that tells you whether your fix-up works.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-3)

### 5.4.4 · Search and in-order walk  `Core`

**Task.** Add `contains(key)` and `inorder()` (a list of keys). Both ignore colour entirely: a red-black tree is still a BST.

**Example.** after inserting `[5, 2, 8, 1]` -> `contains(8)` is `True`; `inorder()` is `[1, 2, 5, 8]`

**Big O.** Search O(log N) guaranteed; in-order O(N).

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-4)

### 5.4.5 · Height stays logarithmic  `Core`

**Task.** Add `height()` and check it never exceeds 2 * log2(N + 1), even after inserting 1..N in sorted order (which makes a plain BST N tall).

**Example.** insert 1..1000 in order -> `height()` is at most `19`, while a plain BST would be `1000` tall

**Big O.** O(N) to measure. The bound follows from the rules: no two reds in a row, and equal black counts on every path.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-5)

### 5.4.6 · Minimum, maximum and successor  `Practice`

**Task.** Add `minimum(node=None)`, `maximum()` and `successor(key)` (the smallest key greater than key, or `None`), using parent links for the successor.

**Example.** keys `[20, 10, 30, 25]` -> `minimum()` `10`, `maximum()` `30`, `successor(20)` `25`, `successor(30)` `None`

**Big O.** O(log N) each.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-6)

### 5.4.8 · Deletion  `Stretch`

**Task.** Add `delete(key)`. Splice out the node (or its successor), remember the colour removed, and if a black node was removed run the four-case "double black" fix-up with rotations and recolouring. Return `False` if the key is absent.

**Example.** insert 1..100, delete every even key -> `validate()` passes and `inorder()` is the odd keys

**Big O.** O(log N), with at most three rotations.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-8)

In [ ]:
# 5.4.1 Nodes, the NIL sentinel and rotations
class RBNode:
    __slots__ = ("key", "color", "left", "right", "parent")

    def __init__(self, key, color, nil=None):
        raise NotImplementedError


class RedBlackTree:
    def __init__(self):
        raise NotImplementedError

    def rotate_left(self, x):
        raise NotImplementedError

    def rotate_right(self, x):
        raise NotImplementedError

    # 5.4.2 Insert with recolouring and rotations
    def insert(self, key):
        raise NotImplementedError

    def _insert_fixup(self, z):
        raise NotImplementedError

    # 5.4.3 Check the red-black rules
    def validate(self):
        raise NotImplementedError

    # 5.4.4 Search and in-order walk
    def contains(self, key):
        raise NotImplementedError

    def inorder(self):
        raise NotImplementedError

    # 5.4.5 Height stays logarithmic
    def height(self):
        raise NotImplementedError

    def height_bound(self):
        raise NotImplementedError

    # 5.4.6 Minimum, maximum and successor
    def minimum(self, node=None):
        raise NotImplementedError

    def maximum(self):
        raise NotImplementedError

    def _find(self, key):
        raise NotImplementedError

    def successor(self, key):
        raise NotImplementedError

    # 5.4.8 Deletion
    def _transplant(self, u, v):
        raise NotImplementedError

    def delete(self, key):
        raise NotImplementedError

    def _delete_fixup(self, x):
        raise NotImplementedError

### 5.4.7 · Count the rotations  `Practice`

**Task.** Write `rotations_for(keys)` returning how many rotations a red-black tree performs while inserting keys in order. Compare sorted, reversed and shuffled input.

**Example.** `rotations_for(range(1000))` -> fewer than 1000 rotations in total (amortized O(1) per insert)

**Big O.** At most 2 rotations per insert; amortized fewer.

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-7)

In [ ]:
# 5.4.7 Count the rotations
def rotations_for(keys):
    raise NotImplementedError

### 5.4.9 · Treap nodes and rotations  `Core`

**Task.** Create `TreapNode(key, priority)` with `left` and `right`, and write `rotate_right(node)` and `rotate_left(node)` that return the new subtree root.

**Example.** rotating right at a node with left child L makes L the subtree root and the old node L's right child

**Big O.** O(1).

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-9) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-9)

In [ ]:
# 5.4.9 Treap nodes and rotations
class TreapNode:
    __slots__ = ("key", "priority", "left", "right")

    def __init__(self, key, priority):
        raise NotImplementedError


def rotate_right(node):
    raise NotImplementedError


def rotate_left(node):
    raise NotImplementedError

### 5.4.10 · Insert, then rotate up  `Core`

**Task.** Write `treap_insert(node, key, rng)` that gives the new key `rng.random()` as its priority, inserts it as a BST leaf, and on the way back up rotates it above any parent with a smaller priority.

**Example.** `rng = random.Random(1); root = None; for k in range(10): root = treap_insert(root, k, rng)` -> a valid treap whose in-order walk is 0..9

**Big O.** O(log N) expected, whatever order the keys arrive in.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-10) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-10)

In [ ]:
# 5.4.10 Insert, then rotate up
def treap_insert(node, key, rng, priority=None):
    raise NotImplementedError

In [ ]:
# Test helpers: run once.
def tree(keys):
    t = RedBlackTree()
    for k in keys:
        t.insert(k)
    return t


def treap(keys, rng):
    root = None
    for k in keys:
        root = treap_insert(root, k, rng)
    return root


def keys_of(node):
    return [] if node is None else keys_of(node.left) + [node.key] + keys_of(node.right)

In [ ]:
def test_ex_5_4_1():
    t = RedBlackTree()
    assert t.root is t.NIL
    assert t.NIL.color == BLACK
    x = RBNode(2, BLACK, t.NIL)
    a, y = (RBNode(1, RED, t.NIL), RBNode(4, RED, t.NIL))
    b, c = (RBNode(3, BLACK, t.NIL), RBNode(5, BLACK, t.NIL))
    x.left, x.right, a.parent, y.parent = (a, y, x, x)
    y.left, y.right, b.parent, c.parent = (b, c, y, y)
    t.root = x
    t.rotate_left(x)
    assert t.root is y
    assert y.left is x
    assert x.parent is y
    assert y.parent is t.NIL
    assert x.left is a
    assert x.right is b
    assert b.parent is x
    assert y.right is c
    t.rotate_right(y)
    assert t.root is x
    assert x.right is y
    assert y.parent is x
    assert y.left is b
    assert b.parent is y
    assert t.rotations == 2


test_ex_5_4_1()
print('5.4.1 passed')

In [ ]:
def test_ex_5_4_2():
    t = tree(range(1, 8))
    assert t.root.key == 2
    assert t.root.color == BLACK
    assert t.size == 7
    assert not t.insert(5)
    assert t.size == 7


test_ex_5_4_2()
print('5.4.2 passed')

In [ ]:
def test_ex_5_4_3():
    t = RedBlackTree()
    for k in range(1, 8):
        t.insert(k)
        t.validate()
    assert tree(range(1, 1001)).validate() > 0


test_ex_5_4_3()
print('5.4.3 passed')

In [ ]:
def test_ex_5_4_4():
    t = tree([5, 2, 8, 1])
    assert t.contains(8)
    assert not t.contains(3)
    assert t.inorder() == [1, 2, 5, 8]
    assert tree(range(1, 8)).inorder() == list(range(1, 8))


test_ex_5_4_4()
print('5.4.4 passed')

In [ ]:
def test_ex_5_4_5():
    assert tree(range(1, 8)).height() == 4
    t = tree(range(1, 1001))
    assert t.height() <= t.height_bound()
    assert t.height() <= 19


test_ex_5_4_5()
print('5.4.5 passed')

In [ ]:
def test_ex_5_4_6():
    t = tree([20, 10, 30, 25])
    assert t.minimum().key == 10
    assert t.maximum() == 30
    assert t.successor(20) == 25
    assert t.successor(30) is None
    assert t.successor(10) == 20


test_ex_5_4_6()
print('5.4.6 passed')

In [ ]:
def test_ex_5_4_7():
    assert rotations_for(range(1000)) < 1000


test_ex_5_4_7()
print('5.4.7 passed')

In [ ]:
def test_ex_5_4_8():
    rng = random.Random(11)
    for _ in range(20):
        t, keys = (RedBlackTree(), set())
        for _ in range(200):
            k = rng.randrange(300)
            if rng.random() < 0.6:
                t.insert(k)
                keys.add(k)
            else:
                assert t.delete(k) == (k in keys)
                keys.discard(k)
            t.validate()
        assert t.inorder() == sorted(keys)
        assert t.size == len(keys)
    t = tree(range(1, 101))
    for k in range(2, 101, 2):
        t.delete(k)
    t.validate()
    assert t.inorder() == list(range(1, 101, 2))


test_ex_5_4_8()
print('5.4.8 passed')

In [ ]:
def test_ex_5_4_9():
    top, left, a, b, c = (TreapNode(5, 0.5), TreapNode(3, 0.9), TreapNode(1, 0.1), TreapNode(4, 0.2), TreapNode(7, 0.3))
    top.left, top.right, left.left, left.right = (left, c, a, b)
    root = rotate_right(top)
    assert root is left
    assert root.right is top
    assert top.left is b
    assert keys_of(root) == [1, 3, 4, 5, 7]
    root = rotate_left(root)
    assert root is top
    assert top.left is left
    assert left.right is b
    assert keys_of(root) == [1, 3, 4, 5, 7]


test_ex_5_4_9()
print('5.4.9 passed')

In [ ]:
def test_ex_5_4_10():
    root = treap(range(10), random.Random(1))
    assert keys_of(root) == list(range(10))
    stack = [root]
    while stack:
        n = stack.pop()
        for child in (n.left, n.right):
            if child:
                assert child.priority <= n.priority
                stack.append(child)


test_ex_5_4_10()
print('5.4.10 passed')

### 5.4.11 · Search  `Core`

**Task.** Write `treap_contains(node, key)`. Priorities play no part in search.

**Example.** after inserting `[5, 2, 8]` -> `treap_contains(root, 8)` is `True`, `treap_contains(root, 3)` is `False`

**Big O.** O(log N) expected.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-11) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-11)

In [ ]:
# 5.4.11 Search
def treap_contains(node, key):
    raise NotImplementedError

In [ ]:
def test_ex_5_4_11():
    root = None
    for k in [5, 2, 8]:
        root = treap_insert(root, k, random.Random(k))
    assert treap_contains(root, 8)
    assert not treap_contains(root, 3)
    assert not treap_contains(None, 1)


test_ex_5_4_11()
print('5.4.11 passed')

### 5.4.12 · Delete by rotating down  `Core`

**Task.** Write `treap_delete(node, key)`: find the node, then rotate it down (always lifting the child with the larger priority) until it has at most one child, and splice it out.

**Example.** insert 0..99, delete the multiples of 3 -> in-order gives the other 66 keys and the treap is still valid

**Big O.** O(log N) expected.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-12) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-12)

In [ ]:
# 5.4.12 Delete by rotating down
def treap_delete(node, key):
    raise NotImplementedError

In [ ]:
def test_ex_5_4_12():
    root = treap(range(100), random.Random(2))
    for k in range(0, 100, 3):
        root = treap_delete(root, k)
    assert keys_of(root) == [k for k in range(100) if k % 3]
    assert keys_of(treap_delete(root, 1000)) == [k for k in range(100) if k % 3]


test_ex_5_4_12()
print('5.4.12 passed')

### 5.4.13 · Validate both orders at once  `Core`

**Task.** Write `is_valid_treap(node)` checking BST order on keys and heap order on priorities in one recursive pass, and `inorder(node)`.

**Example.** any treap built by 5.4.10 -> `True`; swap two priorities by hand -> `False`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-13) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-13)

In [ ]:
# 5.4.13 Validate both orders at once
def is_valid_treap(node, lo=None, hi=None):
    raise NotImplementedError


def inorder(node):
    raise NotImplementedError

In [ ]:
def test_ex_5_4_13():
    root = treap(range(10), random.Random(1))
    assert inorder(root) == list(range(10))
    assert is_valid_treap(root)
    bad = TreapNode(5, 0.1)
    bad.left = TreapNode(2, 0.9)
    assert not is_valid_treap(bad)
    unordered = TreapNode(5, 0.9)
    unordered.left = TreapNode(7, 0.1)
    assert not is_valid_treap(unordered)


test_ex_5_4_13()
print('5.4.13 passed')

### 5.4.14 · Random priorities do the balancing  `Core`

**Task.** Write `treap_height(keys, rng)` building a treap from keys inserted in the given order. Insert 1..1000 in sorted order and compare with a plain BST (7.4.6).

**Example.** sorted 1..1000 with `random.Random(0)` -> height around 20 to 30, versus 1000 for a plain BST

**Big O.** Expected height about 3 x log2 N, the same as a BST built from shuffled keys, so roughly 25 to 30 for N = 1000.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-14) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-14)

In [ ]:
# 5.4.14 Random priorities do the balancing
def treap_height(keys, rng):
    raise NotImplementedError

In [ ]:
def test_ex_5_4_14():
    assert treap_height(range(1, 1001), random.Random(0)) < 40


test_ex_5_4_14()
print('5.4.14 passed')

### 5.4.15 · Split and merge  `Stretch`

**Task.** Write `split(node, key)` returning `(left, right)` treaps with keys < key and >= key, and `merge(a, b)` joining treaps where every key in a is smaller than every key in b. Use them to delete a whole key range.

**Example.** split a treap of 0..9 at 4 -> in-orders `[0, 1, 2, 3]` and `[4, 5, 6, 7, 8, 9]`; merging them restores 0..9

**Big O.** O(log N) expected each. Many contest "implicit treap" tricks are built from these two operations.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-4-15) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-4-15)

In [ ]:
# 5.4.15 Split and merge
def split(node, key):
    raise NotImplementedError


def merge(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_5_4_15():
    root = treap(range(10), random.Random(5))
    a, b = split(root, 4)
    assert keys_of(a) == [0, 1, 2, 3]
    assert keys_of(b) == [4, 5, 6, 7, 8, 9]
    assert is_valid_treap(a)
    assert is_valid_treap(b)
    m = merge(a, b)
    assert keys_of(m) == list(range(10))
    assert is_valid_treap(m)


test_ex_5_4_15()
print('5.4.15 passed')